# Task 3 Playground: Taylor-Saliency Pruning

This notebook demonstrates the implemented Task 3 features: first-order Taylor saliency, a cubic gradual-pruning schedule, global pruning across `Linear` layers, exact target budgets, monotonic masks, Adam-state clearing, and pruning during classifier training.

Run cells from top to bottom with the repository root as the working directory and the project's `.venv` kernel selected. No external dataset or plotting package is required; the notebook prints numeric tables and a learning curve. Pruning only removes connections: regrowth and Part 4 Pareto sweeps are not implemented.

In [2]:
import numpy as np

from nn import Adam, Linear, ReLU, Sequential
from prune import CubicPruningSchedule, PruningResult, TaylorPruner, taylor_saliency
from train.trainer import train_classifier

np.set_printoptions(precision=5, suppress=True)
print("Task 3 pruning playground is ready")

Task 3 pruning playground is ready


## 1. Taylor saliency

For removing a weight `w`, the first-order Taylor loss change is approximately `-w * gradient`. The pruning score is its magnitude, `abs(w * gradient)`: smaller scores are considered less important and are pruned first.

In [3]:
weights = np.array([[2.0, -3.0], [0.5, 4.0]])
gradients = np.array([[-0.25, 0.5], [2.0, -0.1]])
scores = taylor_saliency(weights, gradients)
print("weights:\n", weights)
print("gradients:\n", gradients)
print("|weight * gradient|:\n", scores)
assert np.array_equal(scores, np.abs(weights * gradients))

weights:
 [[ 2.  -3. ]
 [ 0.5  4. ]]
gradients:
 [[-0.25  0.5 ]
 [ 2.   -0.1 ]]
|weight * gradient|:
 [[0.5 1.5]
 [1.  0.4]]


The score helper requires matching shapes and finite values. We can also calculate a score for one connection by hand and check that it agrees.

In [4]:
np.array_equal(scores, np.abs(weights * gradients))

True

In [5]:
i, j = 0, 1
print("hand score:", abs(weights[i, j] * gradients[i, j]))
print("helper score:", scores[i, j])
np.testing.assert_allclose(abs(weights[i, j] * gradients[i, j]), scores[i, j])

for description, bad_weights, bad_gradients in [
    ("shape mismatch", np.ones((2, 2)), np.ones(2)),
    ("non-finite weight", np.array([[np.inf]]), np.ones((1, 1))),
]:
    try:
        taylor_saliency(bad_weights, bad_gradients)
    except ValueError as error:
        print(f"{description}: {error}")

hand score: 1.5
helper score: 1.5
shape mismatch: Weight shape (2, 2) does not match gradient shape (2,)
non-finite weight: Weights must contain only finite values


## 2. Cubic pruning schedule

Between `begin_step` and `end_step`, the target is `final + (initial - final) * (1 - progress)^3`. It stays at the initial value through warm-up, reaches the final value at `end_step`, and remains there afterwards. `should_update` limits when the target is applied; the end step is always included even if it is not aligned to `update_frequency`.

In [6]:
schedule = CubicPruningSchedule(
    initial_sparsity=0.0,
    final_sparsity=0.8,
    begin_step=2,
    end_step=14,
    update_frequency=3,
)

print("step | target sparsity | update?")
print("-----+-----------------+--------")
for step in range(0, 17):
    print(f"{step:4d} | {schedule.sparsity_at(step):15.5f} | {schedule.should_update(step)}")

assert schedule.sparsity_at(0) == 0.0
assert schedule.sparsity_at(2) == 0.0
assert schedule.sparsity_at(14) == 0.8
assert schedule.sparsity_at(100) == 0.8
assert schedule.should_update(2) and schedule.should_update(5)
assert schedule.should_update(14) and not schedule.should_update(15)

step | target sparsity | update?
-----+-----------------+--------
   0 |         0.00000 | False
   1 |         0.00000 | False
   2 |         0.00000 | True
   3 |         0.18380 | False
   4 |         0.33704 | False
   5 |         0.46250 | True
   6 |         0.56296 | False
   7 |         0.64120 | False
   8 |         0.70000 | True
   9 |         0.74213 | False
  10 |         0.77037 | False
  11 |         0.78750 | True
  12 |         0.79630 | False
  13 |         0.79954 | False
  14 |         0.80000 | True
  15 |         0.80000 | False
  16 |         0.80000 | False


The schedule only chooses a target and update step. It does not select weights itself; `TaylorPruner` consumes the target and the current gradients.

In [7]:
begin, end = schedule.begin_step, schedule.end_step
probe_step = 8
progress = (probe_step - begin) / (end - begin)
expected = schedule.final_sparsity * (1 - (1 - progress) ** 3)
print("schedule result:", schedule.sparsity_at(probe_step))
print("formula result:", expected)
np.testing.assert_allclose(schedule.sparsity_at(probe_step), expected)

schedule result: 0.7000000000000001
formula result: 0.7000000000000001


## 3. Hand-checkable global pruning across layers

Set known weights and gradients in two Linear layers. There are eight connections total. At 50% target sparsity, the four globally lowest saliency scores should be removed, even though they are distributed across both layers. We also initialize Adam state first so we can verify that pruning clears moments and per-weight bias-correction ages immediately.

In [8]:
first = Linear(2, 2, activation="linear", rng=np.random.default_rng(1))
second = Linear(2, 2, activation="linear", rng=np.random.default_rng(2))
model = Sequential(first, second)
first.W.data[...] = [[1.0, 2.0], [3.0, 4.0]]
second.W.data[...] = [[5.0, 6.0], [7.0, 8.0]]
first.W.grad = np.array([[0.1, 2.0], [1.0, 0.05]])
second.W.grad = np.array([[0.06, 0.5], [1.0, 0.05]])

first_scores = taylor_saliency(first.W.data, first.W.grad)
second_scores = taylor_saliency(second.W.data, second.W.grad)
print("first layer saliency:\n", first_scores)
print("second layer saliency:\n", second_scores)

optimizer = Adam(model.parameters(), lr=0.01)
for parameter in optimizer.params:
    parameter.grad = np.ones_like(parameter.data)
optimizer.step()

# Set the exact weights and gradients after the warm-up optimizer step, so the
# saliency ordering below remains explicit while Adam has non-empty state.
first.W.data[...] = [[1.0, 2.0], [3.0, 4.0]]
second.W.data[...] = [[5.0, 6.0], [7.0, 8.0]]
first.W.grad = np.array([[0.1, 2.0], [1.0, 0.05]])
second.W.grad = np.array([[0.06, 0.5], [1.0, 0.05]])

pruner = TaylorPruner(model)
result = pruner.step(0.5, optimizer)
print("result:", result)
print("first mask:\n", first.mask)
print("second mask:\n", second.mask)
print("pruned weights are zero:", first.W.data, second.W.data)
assert isinstance(result, PruningResult)
assert result.total_connections == 8 and result.pruned_connections == 4
assert result.active_connections == 4 and result.achieved_sparsity == 0.5
np.testing.assert_array_equal(first.mask, [[0, 1], [1, 0]])
np.testing.assert_array_equal(second.mask, [[0, 1], [1, 0]])

first layer saliency:
 [[0.1 4. ]
 [3.  0.2]]
second layer saliency:
 [[0.3 3. ]
 [7.  0.4]]
result: PruningResult(target_sparsity=0.5, achieved_sparsity=0.5, pruned_connections=4, active_connections=4, total_connections=8)
first mask:
 [[0. 1.]
 [1. 0.]]
second mask:
 [[0. 1.]
 [1. 0.]]
pruned weights are zero: [[0. 2.]
 [3. 0.]] [[0. 6.]
 [7. 0.]]


In [9]:
# Each pruned layer's W is located by object identity in the optimizer list.
for layer in (first, second):
    parameter_index = next(i for i, p in enumerate(optimizer.params) if p is layer.W)
    inactive = layer.mask == 0
    print(
        "inactive W/m/v/age counts:",
        np.count_nonzero(inactive),
        np.count_nonzero(optimizer.m[parameter_index][inactive]),
        np.count_nonzero(optimizer.v[parameter_index][inactive]),
        np.count_nonzero(optimizer.step_counts[parameter_index][inactive]),
    )
    np.testing.assert_array_equal(layer.W.data[inactive], 0.0)
    np.testing.assert_array_equal(optimizer.m[parameter_index][inactive], 0.0)
    np.testing.assert_array_equal(optimizer.v[parameter_index][inactive], 0.0)
    np.testing.assert_array_equal(optimizer.step_counts[parameter_index][inactive], 0)
print("PASS: pruning immediately enforces zero weights and clears Adam state.")

inactive W/m/v/age counts: 2 0 0 0
inactive W/m/v/age counts: 2 0 0 0
PASS: pruning immediately enforces zero weights and clears Adam state.


## 4. Exact budgets, stable ties, and no regrowth

The pruner counts all Linear weights globally and uses `int(total_connections * target_sparsity)` as its zero budget. Existing zeros stay pruned. When scores tie, stable sorting selects earlier flattened connections first. Requesting a sparsity below the current value raises an error instead of silently regrowing weights.

In [10]:
tie_layer = Linear(2, 2, activation="linear", rng=np.random.default_rng(4))
tie_model = Sequential(tie_layer)
tie_layer.W.data.fill(1.0)
tie_layer.W.grad = np.array([[0.1, 0.1], [0.2, 0.3]])
tie_result = TaylorPruner(tie_model).step(0.25, Adam(tie_model.parameters()))
print("equal lowest scores prune first flattened coordinate:", tie_layer.mask)
assert tie_result.pruned_connections == 1
np.testing.assert_array_equal(tie_layer.mask, [[0, 1], [1, 1]])

try:
    TaylorPruner(model).step(0.25, optimizer)
except ValueError as error:
    print("regrowth request rejected:", error)
else:
    raise AssertionError("Expected lower sparsity target to be rejected")

equal lowest scores prune first flattened coordinate: [[0. 1.]
 [1. 1.]]
regrowth request rejected: target_sparsity is below current sparsity; TaylorPruner does not regrow connections


## 5. Scheduled pruning during training

`train_classifier` requires both `pruning_schedule` and `pruner`, and verifies the pruner belongs to the trained model. At the end of an epoch whose number satisfies `should_update(epoch)`, it uses the last mini-batch's Linear weight gradients to choose connections, applies the target, and records the resulting global masked fraction in `EpochMetrics.sparsity`.

This example uses 40 Linear weights and a schedule that updates at epochs 1, 3, 5, 7, and 8. `update_frequency=2` means every second step relative to `begin_step`, with `end_step` forced as an update.

In [11]:
data_rng = np.random.default_rng(2)
centers = np.array([[-2.0, -1.0], [2.0, -1.0], [0.0, 2.0]])
features = np.concatenate([
    center + data_rng.normal(0.0, 0.5, size=(60, 2)) for center in centers
])
labels = np.repeat(np.arange(3), 60)
order = data_rng.permutation(len(labels))
features, labels = features[order], labels[order]

model_rng = np.random.default_rng(7)
classifier = Sequential(
    Linear(2, 8, activation="relu", rng=model_rng),
    ReLU(),
    Linear(8, 3, activation="linear", rng=model_rng),
)
schedule = CubicPruningSchedule(
    initial_sparsity=0.0,
    final_sparsity=0.6,
    begin_step=1,
    end_step=8,
    update_frequency=2,
)
optimizer = Adam(classifier.parameters(), lr=0.03, weight_decay=1e-4)
pruner = TaylorPruner(classifier)

history = train_classifier(
    classifier,
    optimizer,
    features,
    labels,
    epochs=8,
    batch_size=32,
    seed=5,
    pruning_schedule=schedule,
    pruner=pruner,
)

total_weights = sum(layer.W.data.size for layer in pruner.layers)
print("Linear connections:", total_weights)
print("epoch | loss     | accuracy | schedule target | measured sparsity")
print("------+----------+----------+-----------------+------------------")
for row in history:
    print(
        f"{row.epoch:5d} | {row.loss:8.5f} | {row.accuracy:8.3f} | "
        f"{schedule.sparsity_at(row.epoch):15.5f} | {row.sparsity:16.3f}"
    )

Linear connections: 40
epoch | loss     | accuracy | schedule target | measured sparsity
------+----------+----------+-----------------+------------------
    1 |  0.67878 |    0.706 |         0.00000 |            0.000
    2 |  0.25109 |    0.956 |         0.22216 |            0.000
    3 |  0.09901 |    1.000 |         0.38134 |            0.375
    4 |  0.06374 |    0.994 |         0.48805 |            0.375
    5 |  0.02792 |    1.000 |         0.55277 |            0.550
    6 |  0.02723 |    1.000 |         0.58601 |            0.550
    7 |  0.01695 |    1.000 |         0.59825 |            0.575
    8 |  0.02213 |    1.000 |         0.60000 |            0.600


In [12]:
sparsity_curve = np.array([item.sparsity for item in history])
loss_curve = np.array([item.loss for item in history])
accuracy_curve = np.array([item.accuracy for item in history])
print("sparsity by epoch:", sparsity_curve)
print("loss finite:", np.isfinite(loss_curve).all())
print("accuracy finite:", np.isfinite(accuracy_curve).all())
print("final pruner sparsity:", pruner.sparsity)
print("final active/total:", sum(np.count_nonzero(layer.mask) for layer in pruner.layers), "/", total_weights)

assert np.isfinite(loss_curve).all() and np.isfinite(accuracy_curve).all()
assert np.all(np.diff(sparsity_curve) >= 0)
np.testing.assert_allclose(sparsity_curve, [0.0, 0.0, 0.375, 0.375, 0.55, 0.55, 0.575, 0.6])
assert pruner.sparsity == 0.6
for layer in pruner.layers:
    inactive = layer.mask == 0
    np.testing.assert_array_equal(layer.W.data[inactive], 0.0)
print("PASS: scheduled updates, exact global sparsity, monotonic pruning, finite training.")

sparsity by epoch: [0.    0.    0.375 0.375 0.55  0.55  0.575 0.6  ]
loss finite: True
accuracy finite: True
final pruner sparsity: 0.6
final active/total: 16 / 40
PASS: scheduled updates, exact global sparsity, monotonic pruning, finite training.


## 6. Interpretation and boundaries

- Taylor pruning ranks all currently active `Linear.W` entries together, not layer by layer.
- A target is converted to an integer zero count, so achieved sparsity is exact at the connection-count granularity and may differ slightly from a non-representable fraction.
- Masks are monotonic: pruned connections are not candidates for later pruning and cannot be revived by lowering the target.
- Pruning sets weights to zero and immediately clears the corresponding Adam first moment, second moment, and per-entry bias-correction age.
- The reported epoch loss/accuracy aggregate the minibatches before that epoch's end-of-epoch pruning update. The next epoch trains using the newly pruned model.
- This is dense NumPy computation with masked values; it demonstrates correctness but does not claim sparse-compute speedup.
- Connection regrowth and Part 4 Pareto sweeps are outside the implemented Task 3 API.

## Your turn

1. Change the final sparsity and compare the selected lowest-saliency connections.
2. Try different update frequencies and confirm the final schedule step still updates.
3. Add another Linear layer and verify the target budget is applied globally across all layers.
4. Change the random seeds and inspect whether the loss, accuracy, and pruning pattern remain finite and reproducible.
5. Try requesting a target below existing sparsity and inspect the no-regrowth error.